In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_orders_df = spark.table("ecommerce_lakehouse.bronze.orders_raw")
display(bronze_orders_df)

In [0]:
silver_orders_df = bronze_orders_df \
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    ) \
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    ) \
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    ) \
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    ) \
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    ) \
    .dropDuplicates(["order_id"]) \
    .withColumn(
        "purchase_date",
        to_date("order_purchase_timestamp")
    ) \
    .withColumn(
        "purchase_year",
        year("order_purchase_timestamp")
    ) \
    .withColumn(
        "purchase_month",
        month("order_purchase_timestamp")
    ) \
    .withColumn(
        "delivery_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        )
    ) \
    .withColumn(
        "is_delivered",
        when(
            col("order_status") == "delivered",
            True
        ).otherwise(False)
    )

In [0]:
silver_orders_df.createOrReplaceTempView(
    "silver_orders_updates"
)

In [0]:
%sql
MERGE INTO ecommerce_lakehouse.silver.orders_clean AS target
USING silver_orders_updates AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *